<a href="https://colab.research.google.com/github/FranciscoBPereira/AnaliseDados_MEI_2526/blob/main/AD2526_Aula10_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**0 - Install and Preprocessing**

In [ ]:
#Install libraries to run this notebook.

!pip install transformers sentence-transformers accelerate evaluate



In [ ]:

!pip install datasets

**Dataset Loading and Tokenization**

In [ ]:
# Load Rotten Tomatoes dataset: https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes

from datasets import load_dataset

# Prepare data and splits
tomatoes = load_dataset('rotten_tomatoes')

#tomatoes = load_dataset('cornell-movie-review-data/rotten_tomatoes')

train_data, test_data = tomatoes["train"], tomatoes["test"]

In [ ]:
# Inspect dataset details

tomatoes

In [ ]:
# Print some reviews

for i in range(5):
    print(train_data[i])

In [ ]:
#Model selection

model_id = "bert-base-cased"

In [ ]:
# Tokenization

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)

def preprocess_function(examples):
   """Tokenize input data"""
   return tokenizer(examples["text"], truncation=True)

# Tokenize train/test data
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)


**A. ModelA loading**

In [ ]:
# Load ModelA - This model will just adjust its classification head

from transformers import AutoModelForSequenceClassification

modelA = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)


In [ ]:
# Print layer names for modelA
for name, param in modelA.named_parameters():
    print(name)

In [ ]:
# Freeze all weights from pretrained BERT module

for name, param in modelA.named_parameters():

     # Trainable classification head
     if name.startswith("classifier"):
        param.requires_grad = True

      # Freeze everything else
     else:
        param.requires_grad = False

In [ ]:
# We can check whether the model was correctly updated

for name, param in modelA.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")

In [ ]:
# Define training arguments

from transformers import TrainingArguments, DataCollatorWithPadding

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Training arguments for parameter tuning
training_args = TrainingArguments(
   "model",
   learning_rate=2e-5,
   per_device_train_batch_size=16,
   per_device_eval_batch_size=16,
   num_train_epochs=1,
   weight_decay=0.01,
   save_strategy="epoch",
   report_to="none"
)

In [ ]:
# Define evaluation metrics
import numpy as np
import evaluate


def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

**B. ModelA Classification Head Training**

In [ ]:
from transformers import Trainer

# Trainer which executes the training process
trainerA = Trainer(
   model=modelA,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)
trainerA.train()

**C. ModelA Evaluation**

In [ ]:
trainerA.evaluate()


**D. ModelB Loading**

In [1]:
# Load ModelB - We will perform fine-tuning with this model

modelB = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)


NameError: name 'AutoModelForSequenceClassification' is not defined

In [ ]:
# We can check that all the weights in this model will be adjusted

for name, param in modelB.named_parameters():
     print(f"Parameter: {name} ----- {param.requires_grad}")

**E. ModelB Fine-tuning**

In [ ]:

# Trainer which executes the training process
trainerB = Trainer(
   model=modelB,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)



trainerB.train()

**F. ModelB Evaluation**

In [ ]:
trainerB.evaluate()

**Analyse results**

1. Time to train

2. Effectiveness on test set